# Gate 28: Proxy Spectral-Gap Audit

No-extra-light-state target: audit whether a proxy lift can clear the dimensionless Hopf-gap target `mu_H = 64 pi^5` for modes in `H_perp`.

This notebook uses the supplied Berger scalar spectrum proxy as a temporary proxy for `D_hat^dagger D_hat`. It is not the final twisted Dirac spectrum and does not prove the no-extra-light-state theorem.

In [ ]:
from pathlib import Path
import sys

root = Path.cwd()
if not (root / 'src').exists():
    root = root.parent
sys.path.insert(0, str(root / 'src'))

import numpy as np

from positivity import (
    compensated_barrier,
    complement_projector,
    diagonal_proxy_operator,
    finite_berger_modes,
    gap_condition_with_operator,
    orthogonal_projector,
    psd_barrier_from_q,
    restrict_to_complement,
    zero_mode_basis_from_modes,
)
from spectral_gap import (
    MU_H,
    alpha_scaled_a,
    curvature_profile_term,
    natural_lambda2,
    positive_barrier,
    required_lambda2,
    scan_gap_robustness,
    scan_proxy_spectrum,
    scan_with_profile_terms,
)

MU_H

## Scan Proxy Berger Modes

Scan `0 <= k <= n_max`, `0 <= j <= k` with `n_max = 20`. The zero mode is excluded when reporting the first nonzero proxy eigenvalue.

In [ ]:
n_max = 20
a = 1.0
reference_lambda2 = 0.1
scan = scan_proxy_spectrum(a=a, n_max=n_max, lambda2=reference_lambda2, exclude_zero=True)
first_nonzero = scan[0]
first_nonzero

## Heat-Lift Choices

Each `Lambda^2` value is fixed explicitly. The audit reports pass/fail against `mu_H`; it does not tune `Lambda^2` silently.

In [ ]:
lambda2_choices = [0.01, 0.1, 1.0, 10.0]
rows = []
for lambda2 in lambda2_choices:
    lifted_scan = scan_proxy_spectrum(a=a, n_max=n_max, lambda2=lambda2, exclude_zero=True)
    first = lifted_scan[0]
    rows.append({
        'lambda2': lambda2,
        'first_mode': (first['k'], first['j']),
        'first_unlifted_d': first['d'],
        'first_lifted': first['lifted'],
        'required_lambda2_for_first_mode': required_lambda2(first['d']),
        'passes_mu_H': first['passes'],
    })
rows

## Naturalness Audit

Gate 28B fixes the heat-kernel cutoff to the universal overlap width `Lambda^2 = S = 1/(4 pi)`. Other cutoff values are sensitivity scans only.

In [ ]:
natural = natural_lambda2()
first_natural = scan_proxy_spectrum(a=1.0, n_max=20, exclude_zero=True)[0]
lambda2_max = required_lambda2(first_natural['d'])
{
    'first_mode': (first_natural['k'], first_natural['j']),
    'first_unlifted_d': first_natural['d'],
    'lambda2_max': lambda2_max,
    'natural_lambda2': natural,
    'natural_is_below_max': natural <= lambda2_max,
    'passes_at_natural_width': first_natural['passes'],
}

In [ ]:
a_values = [0.573, 1.0, alpha_scaled_a()]
n_max_values = [10, 20, 40]
v_min_values = [0.0, -0.01 * MU_H, -0.05 * MU_H, -0.10 * MU_H]
robustness_rows = scan_gap_robustness(
    a_values=a_values,
    n_max_values=n_max_values,
    v_min_values=v_min_values,
)
robustness_rows

## Gate 28C — Curvature/Profile Positivity Audit

Gate 28C audits the explicit condition `gamma_hat K[rho_Phi] + V_hat_T(y) >= 0` on `H_perp` in proxy form. The models below do not add new theory; they are executable sign tests for zero, positive, negative, and compensated curvature/profile terms.

In [ ]:
# Gate 28B failure table for negative V_min values.
[row for row in robustness_rows if row['a'] == 0.573 and row['n_max'] == 10]

In [ ]:
profile_model_definitions = {
    'zero': 'V = 0',
    'positive_barrier': 'V = strength * lambda_{k,j}^power >= 0',
    'bounded_negative': 'V = -depth * mu_H',
    'compensated': 'V = negative well + positive barrier',
}
profile_model_definitions

In [ ]:
first_worst = scan_proxy_spectrum(a=0.573, n_max=10, exclude_zero=True)[0]
worst_zero_margin = first_worst['lifted'] - MU_H
compensation_needed = {
    '-0.01*mu_H': max(0.0, 0.01 * MU_H - worst_zero_margin),
    '-0.05*mu_H': max(0.0, 0.05 * MU_H - worst_zero_margin),
    '-0.10*mu_H': max(0.0, 0.10 * MU_H - worst_zero_margin),
}
compensation_needed

In [ ]:
target_comp = compensation_needed['-0.10*mu_H']
barrier_basis = positive_barrier(first_worst['k'], first_worst['j'], strength=1.0, power=1.0)
required_barrier_strength = target_comp / barrier_basis
required_barrier_strength

In [ ]:
profile_rows = scan_with_profile_terms(
    a_values=[0.573, 1.0, alpha_scaled_a()],
    n_max_values=[10, 20, 40],
    profile_models=[
        'zero',
        {'name': 'positive_barrier', 'model': 'positive_barrier', 'params': {'strength': 1.0, 'power': 1.0}},
        {'name': 'bounded_negative_1pct', 'model': 'bounded_negative', 'params': {'depth': 0.01}},
        {'name': 'bounded_negative_10pct', 'model': 'bounded_negative', 'params': {'depth': 0.10}},
        {'name': 'compensated_worst_10pct', 'model': 'compensated', 'params': {'depth': 0.10, 'width': 1e12, 'strength': required_barrier_strength, 'power': 1.0}},
    ],
)
profile_rows

## Gate 28D — Positive-Semidefinite Profile Construction

Gate 28D formalizes the sufficient proxy condition as a finite-basis positive-semidefinite contribution on `H_perp`: `Q^dagger Q`, or `Q^dagger Q - C Pi_0` with the subtraction confined to the protected zero-mode subspace.

In [ ]:
modes_28d = finite_berger_modes(n_max=4)
zero_basis = zero_mode_basis_from_modes(modes_28d)
base_operator = diagonal_proxy_operator(modes_28d)
p0 = orthogonal_projector(zero_basis)
p_perp = complement_projector(zero_basis)
{
    'basis_size': len(modes_28d),
    'zero_mode_count': zero_basis.shape[1],
    'zero_projector_trace': float(np.trace(p0)),
    'complement_projector_trace': float(np.trace(p_perp)),
    'projectors_orthogonal': bool(np.allclose(p0 @ p_perp, np.zeros_like(p0))),
}

In [ ]:
dimension = base_operator.shape[0]
zero_profile = np.zeros_like(base_operator)
psd_profile = psd_barrier_from_q(np.diag([0.0] + [2.0] * (dimension - 1)))
negative_well = np.diag([0.0] + [-0.01 * MU_H] * (dimension - 1))
compensating_q = np.diag([0.0] + [np.sqrt(0.01 * MU_H)] * (dimension - 1))
compensated_profile = negative_well + psd_barrier_from_q(compensating_q)
shifted_zero_profile = compensated_barrier(compensating_q, zero_basis, zero_mode_shift=5.0)

profile_operator_rows = []
for name, profile in [
    ('zero', zero_profile),
    ('psd_barrier', psd_profile),
    ('negative_well_1pct', negative_well),
    ('compensated_barrier', compensated_profile),
    ('qdagq_minus_c_pi0', shifted_zero_profile),
]:
    result = gap_condition_with_operator(base_operator, profile, zero_basis, MU_H)
    profile_operator_rows.append({'model': name, **result})
profile_operator_rows

In [ ]:
zero_vector = zero_basis[:, 0]
{
    'complement_kills_zero_mode': bool(np.allclose(p_perp @ zero_vector, np.zeros_like(zero_vector))),
    'complement_only_barrier_kills_zero_mode': bool(np.allclose(psd_profile @ zero_vector, np.zeros_like(zero_vector))),
    'restricted_shifted_zero_min': float(np.linalg.eigvalsh(restrict_to_complement(shifted_zero_profile, zero_basis))[0]),
}

## Limitation

This is a proxy audit only. The full twisted Dirac `H_T` spectrum remains open, and the no-extra-light-state theorem is not proven by this notebook.